In [3]:
import subprocess
import sys

# -----------------------------
# Install Required Libraries
# -----------------------------

libraries = [
    "numpy",
    "scikit-learn",
    "tensorflow",
    "scikeras",
    "matplotlib",
    "pillow"
]

print("Installing required libraries...\n")

for lib in libraries:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])
        print(f"{lib} installed successfully!")
    except Exception as e:
        print(f"Failed to install {lib}: {e}")

print("\nAll required libraries installed successfully!\n")


# -----------------------------
# Imports for Your Project
# -----------------------------

import numpy as np

from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, ClassifierMixin

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Modern wrapper for sklearn + keras
from scikeras.wrappers import KerasClassifier


print("\nAll libraries imported successfully!")
print("Environment is ready for training your CNN model.")

Installing required libraries...

numpy installed successfully!
scikit-learn installed successfully!
tensorflow installed successfully!
scikeras installed successfully!
matplotlib installed successfully!
pillow installed successfully!

All required libraries installed successfully!


All libraries imported successfully!
Environment is ready for training your CNN model.


In [4]:
# Define image dimensions and batch size
img_height, img_width = 128, 128
batch_size = 32
# Define directories for training and testing data
train_data_dir = "../dataset/train"
test_data_dir = "../dataset/test"

In [5]:
# Data augmentation for training images
train_datagen = ImageDataGenerator(
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rescale=1./255  # Normalize pixel values
)

# Data augmentation for testing images (only rescale)
test_datagen = ImageDataGenerator(rescale=1./255)


In [6]:
# Create data generators for training and testing
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # No need to shuffle for evaluation
)



Found 1174 images belonging to 14 classes.
Found 428 images belonging to 14 classes.


In [7]:
# Define the VGG16 base model
def create_InceptionV3_model():
    InceptionV3_model = InceptionV3(
        include_top=False, 
        input_shape=(img_height, img_width, 3)
        )
    InceptionV3_model.trainable = False

    model = Sequential([
        InceptionV3_model,
        GlobalAveragePooling2D(),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(14, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    print(model.summary())
    return model




In [8]:
#MODEL fit
model = create_InceptionV3_model()
model.fit(train_generator, epochs=15, validation_data=test_generator)


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 28s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inception_v3 (Functional)       │ (None, 2, 2, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       131,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_94          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14)             │           910 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,935,086 (83.68 MB)

 Trainable params: 132,174 (516.30 KB)

 Non-trainable params: 21,802,912 (83.17 MB)

None
Epoch 1/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 15s 256ms/step - accuracy: 0.3535 - loss: 2.0547 - val_accuracy: 0.3762 - val_loss: 1.8489
Epoch 2/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 202ms/step - accuracy: 0.5579 - loss: 1.3211 - val_accuracy: 0.5607 - val_loss: 1.3587
Epoch 3/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 198ms/step - accuracy: 0.6397 - loss: 1.1245 - val_accuracy: 0.6519 - val_loss: 1.1003
Epoch 4/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 202ms/step - accuracy: 0.6959 - loss: 0.9684 - val_accuracy: 0.5678 - val_loss: 1.2164
Epoch 5/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 203ms/step - accuracy: 0.7496 - loss: 0.8306 - val_accuracy: 0.6425 - val_loss: 1.0946
Epoch 6/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 200ms/step - accuracy: 0.7675 - loss: 0.7549 - val_accuracy: 0.6472 - val_loss: 1.0726
Epoch 7/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 195ms/step - accuracy: 0.7871 - loss: 0.7001 - val_accuracy: 0.6706 - val_loss: 0.9454
Epoch 8/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 193ms/step - accuracy: 0.8024 - loss: 0.6332 - val_accura

In [9]:
# predicting train dataset 
pridiction = model.predict(test_generator)





14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 239ms/step


In [10]:
#calculate accuracy
from sklearn.metrics import accuracy_score
y_pred = np.argmax(pridiction, axis=1)
y_true = test_generator.classes
accuracy = accuracy_score(y_true, y_pred)
print(accuracy)

0.7476635514018691


In [11]:
# Save the trained model
model.save("../model_saved_files/InceptionV3.h5")
print("Model saved successfully!")

Model saved successfully!
